In [1]:
import base64
import json
import numpy as np
import os
import random
import re
import csv
import time
import urllib.error
import urllib.parse
import urllib.request
import traceback
import random
from pathlib import Path

import pandas as pd
from tqdm.notebook import tqdm


In [2]:
#INPUT_FAKES = ['generated_fakes_with_commons_matches_with_topics_with_atrib5271961625.csv', 'generated_fakes_with_commons_matches_with_topics_with_atrib6736435178.csv', 'generated_fakes_with_commons_matches_with_topics_with_atrib_8499734541.csv']
#INPUT_REALS = ['r2_articles_with_attrib_filter_scored.csv']

INPUT_FAKES = []
INPUT_REALS = ['articles_with_attrib_filter_scored_safe.csv']

fake_dfs = []
for ifake in INPUT_FAKES:
    df = pd.read_csv(ifake, dtype=str).fillna("")
    fake_dfs.append(df)

real_dfs = []
for ireal in INPUT_REALS:
    df = pd.read_csv(ireal, dtype=str).fillna("")
    real_dfs.append(df)


In [44]:
 
# Culture sub-topics that map to "Society" rather than "Entertainment".
_CULTURE_SOCIETY_SUBTOPICS = frozenset({
    'biography', 'women', 'food and drink', 'internet culture',
    'linguistics', 'philosophy and religion', 'architecture',
    'visual arts.fashion', 'visual arts.architecture', 
})
 
# STEM sub-topic -> sub-category overrides; everything else -> "Physical Science".
_STEM_SUBCATEGORY = {
    'stem*' : 'Physical Science',
    'stem' : 'Physical Science',    
    'chemistry': 'Physical Science',
    'computing': 'Physical Science',
    'engineering': 'Physical Science',
    'mathematics': 'Physical Science',
    'physics': 'Physical Science',
    'technology': 'Physical Science',
    'biology': 'Earth',
    'earth and environment': 'Earth',
    'medicine and health': 'Earth',
    'medicine & health': 'Earth',
    'space': 'Earth',
    'libraries and information': 'History / Society',
}

_PEOPLE_CULTURE_SUBTOPICS = {
    'biography', 'biography*', 'biography.women', 'food and drink', 'internet culture', 'linguistics',
    'philosophy and religion', 'visual arts.architecture', 'architecture', 'fashion', 'biography.biography'
}
 
 
def _normalize(value):
    s = '' if value is None else value
    s = s.lower()
    s = s.replace('&', ' and ')
    s = re.sub(r'\*+', '', s)      # strip '*' markers
    s = re.sub(r'\s+', ' ', s)     # collapse whitespace
    s = s.replace('_', ' ')        # underscores -> spaces
    s = s.replace('  ', ' ')
    return s.strip()
 
 
def _split_combined(combined):
    s = '' if combined is None else combined
    idx = s.find('.')
    if idx >= 0:
        return s[:idx], s[idx + 1:]
    return combined, ''
 
 
def ml_category(topic, subtopic=None):
    if subtopic is None:                       # combined-string mode
        topic, subtopic = _split_combined(topic)
    t = _normalize(topic)
    s = _normalize(subtopic)
 
    if t == 'culture':
        return 'The Human'
    if t == 'geography':
        return 'The World'
    if t == 'history and society':
        return 'The Human'
    if t == 'stem':
        if s in ('medicine and health', 'biology', 'space', 'earth and environment'):
            return 'The World'
        if s == 'libraries and information':
            return 'The Human'
        return 'The Sciences'   # chemistry, computing, engineering, mathematics,
                                # medicine & health, physics, technology, STEM*, unknown
    return None                 # unrecognized topic
 
 
def ml_subcategory(topic, subtopic=None):
    if subtopic is None:                       # combined-string mode
        topic, subtopic = _split_combined(topic)
    cat = ml_category(topic, subtopic)
        
    t = _normalize(topic)
    s = _normalize(subtopic)
 
    if t == 'culture':
        if s == "sports":
            return "Sports"
        return 'People / Culture' if s in _PEOPLE_CULTURE_SUBTOPICS else 'Media'
    if t == 'geography':
        return 'Earth'
    if t == 'history and society':
        return 'History / Society'
    if t == 'stem':
        return _STEM_SUBCATEGORY[s.lower()]
    return None               


In [45]:
def normalize_real(df):
    return (df
        .rename(columns={"name": "title", "sentence_1": "first_sentence",
                          "sentence_2": "second_sentence", "sentence_3": "third_sentence"})
        [["qid", "title", "topic", "first_sentence", "second_sentence", "third_sentence", "image_url", "percentile", "image_license", "image_credit", "image_attribution_url", "flag_score"]]
        .assign(real=True, percentile=lambda df: df['percentile'].astype(float), category=df.topic.map(ml_category), sub_category=df.topic.map(ml_subcategory))
    )[df.qid != ""]

def normalize_fake(df):
    return (df
        .rename(columns={"commons_file_url": "image_url"})
        [["title", "topic", "first_sentence", "second_sentence", "third_sentence", "image_url", "image_license", "image_credit", "image_attribution_url"]]
        .assign(category=df.topic.map(ml_category), sub_category=df.topic.map(ml_subcategory),real=False, flag_score=0)
    )

combined_df = pd.concat(
    [normalize_real(df) for df in real_dfs] +
    [normalize_fake(df) for df in fake_dfs],
    ignore_index=True
)
mask = ~combined_df.real
combined_df.loc[mask, 'percentile'] = np.random.uniform(0, 0.96, mask.sum())
combined_df.loc[mask, 'qid'] = [f"Q-{random.randint(0, 999999999)}" for i in range(mask.sum())]

In [46]:
combined_df.to_csv("real_dump.csv", index=False)

CHUNK_SIZE = 3000

for i, chunk in enumerate(range(0, len(combined_df), CHUNK_SIZE)):
    combined_df.iloc[chunk:chunk + CHUNK_SIZE].to_csv(f"real_dump_{i:04d}.csv", index=False)


In [99]:
combined_df

,title,topic,first_sentence,second_sentence,third_sentence,image_url,image_license,image_credit,image_attribution_url,category,sub_category,real,flag_score,percentile,qid
0,Rajesh Vora,Culture.Biography,Rajesh Vora (born 1985) is an Indian theater a...,He studied dramatic arts at the Maharaja Sayaj...,Vora's performance in the 2012 musical adaptat...,https://upload.wikimedia.org/wikipedia/commons...,PDM,Jack E. Boucher,https://commons.wikimedia.org/w/index.php?titl...,The Human,Society,False,0,0.916053,Q-42005457
1,Polycinematic acid,STEM.Chemistry,Polycinematic acid is a heat-resistant synthet...,"Developed by French chemists, the compound for...","Today, it is widely utilized in both the enter...",https://upload.wikimedia.org/wikipedia/commons...,CC BY-SA 4.0,Colin,https://commons.wikimedia.org/w/index.php?titl...,The Sciences,Physical Science,False,0,0.931863,Q-328189421
2,Redstone Gorge State Park,Geography.Regions.Americas,Redstone Gorge State Park is a nature reserve ...,"Established in 1924, the park features sandsto...",It has served as the backdrop for various hist...,https://upload.wikimedia.org/wikipedia/commons...,PDM,Bureau of Land Management Oregon and Washington,https://commons.wikimedia.org/w/index.php?titl...,The World,Earth,False,0,0.439603,Q-260994579
3,Charles Sterling,Culture.Media,Charles Sterling (1901–1965) was an English pl...,"Known for his sophisticated wit, Sterling wrot...",His productions were filmed live in London and...,https://upload.wikimedia.org/wikipedia/commons...,PDM,Unknown author,https://commons.wikimedia.org/w/index.php?titl...,The Human,Entertainment,False,0,0.669527,Q-229147023
4,Inis Morba,Geography.Regions.Europe,Inis Morba is a small uninhabited island off t...,The island covers an area of approximately two...,"Today, it is protected as a national nature re...",https://upload.wikimedia.org/wikipedia/commons...,PDM,U. S. Fish and Wildlife Service - Northeast Re...,https://commons.wikimedia.org/w/index.php?titl...,The World,Earth,False,0,0.700001,Q-934142100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,Chloe Sterling,Culture.Biography,Chloe Sterling is a British choreographer and ...,She is widely recognized for her innovative mu...,"In 2021, she founded a dance academy in London...",https://upload.wikimedia.org/wikipedia/commons...,CC0,Brooke Cagle brookecagle,https://commons.wikimedia.org/w/index.php?titl...,The Human,Society,False,0,0.845150,Q-63067010
146,Jari Talvi,Culture.Sports,Jari Talvi is a retired Finnish professional f...,"Known for his incredible speed, Talvi scored o...","He later transitioned to coaching, leading his...",https://upload.wikimedia.org/wikipedia/commons...,No Restrictions,UBC Library Digitization Centre,https://commons.wikimedia.org/w/index.php?titl...,The Human,Entertainment,False,0,0.646864,Q-685166648
147,Solomon Cryptological Museum,Culture.Philosophy_and_religion,The Solomon Cryptological Museum is an indepen...,"Established in 1982, the facility conducts adv...",The museum's collection includes thousands of ...,https://upload.wikimedia.org/wikipedia/commons...,CC BY 3.0,Marlene Oostryck (Wiki Takes Fremantle partici...,https://commons.wikimedia.org/w/index.php?titl...,The Human,Society,False,0,0.302526,Q-25492391
148,Ontario Aero-Sentinel,History_and_Society.Military_and_warfare,The Ontario Aero-Sentinel is a military drone ...,"Built by Dominion Defense Industries, the airc...",The drone is primarily used by the Royal Canad...,https://upload.wikimedia.org/wikipedia/commons...,CC BY 4.0,АрміяInform,https://commons.wikimedia.org/w/index.php?titl...,The Human,Society,False,0,0.840622,Q-308214795


In [43]:
ml_subcategory('Culture.Sports')

'Sports'